# Task 5: Exploratory Data Analysis (EDA) – Titanic Dataset

**Objective:** Explore passenger data using statistics and visualizations, identify patterns, trends, relationships and anomalies.

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn.

## How to run
1. Download the Kaggle Titanic `train.csv` dataset.
2. Rename it to **`titanic.csv`** and place it in the same folder as this notebook.
3. Run all cells from top to bottom.

The notebook also attempts to load the dataset from the public GitHub copy if the local CSV is not found. The PDF report accompanying this notebook summarizes the standard Kaggle training dataset (891 rows).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

try:
    df = pd.read_csv("titanic.csv")
    print("Loaded local titanic.csv")
except FileNotFoundError:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    print("Loaded Titanic CSV from the public dataset URL")

df.head()

## 1. Dataset overview
Inspect the number of rows/columns, data types, missing values and duplicate records.

In [ ]:
print("Shape:", df.shape)
display(df.head())
df.info()
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nMissing values:\\n", df.isnull().sum().sort_values(ascending=False))

## 2. Descriptive statistics
Use `.describe()` for numeric columns and `.value_counts()` for categorical columns.

In [ ]:
display(df.describe(include="all").T)
for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    if col in df.columns:
        print(f"\\n{col} value counts:")
        print(df[col].value_counts(dropna=False))

## 3. Data preparation
Create a working copy, fill missing Age with the median, Embarked with the mode, and Fare with the median. Cabin has substantial missingness, so it is not imputed.

In [ ]:
eda = df.copy()
if "Age" in eda.columns:
    eda["Age"] = eda["Age"].fillna(eda["Age"].median())
if "Embarked" in eda.columns:
    eda["Embarked"] = eda["Embarked"].fillna(eda["Embarked"].mode()[0])
if "Fare" in eda.columns:
    eda["Fare"] = eda["Fare"].fillna(eda["Fare"].median())

print("Missing values after selected imputations:")
print(eda.isnull().sum().sort_values(ascending=False).head(10))

## 4. Univariate analysis
Histograms show distributions; count plots show category frequencies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(eda["Age"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Passenger Age Distribution")
sns.countplot(data=eda, x="Survived", ax=axes[1])
axes[1].set_title("Survival Count (0 = No, 1 = Yes)")
plt.tight_layout()
plt.show()

for col in ["Sex", "Pclass", "Embarked"]:
    if col in eda.columns:
        plt.figure(figsize=(7, 4))
        sns.countplot(data=eda, x=col)
        plt.title(f"Count of passengers by {col}")
        plt.tight_layout()
        plt.show()

**Observations**
- The age distribution is concentrated among young adults, with fewer very young children and older passengers.
- In the Kaggle training sample, non-survivors outnumber survivors.
- Third class contains the largest number of passengers; male passengers are more numerous than female passengers.

## 5. Bivariate analysis
Compare survival with sex, passenger class and age.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=eda, x="Sex", y="Survived", errorbar=None, ax=axes[0])
axes[0].set_title("Survival Rate by Sex")
axes[0].set_ylabel("Survival rate")
sns.barplot(data=eda, x="Pclass", y="Survived", errorbar=None, ax=axes[1])
axes[1].set_title("Survival Rate by Passenger Class")
axes[1].set_ylabel("Survival rate")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(data=eda, x="Survived", y="Age")
plt.title("Age Distribution by Survival")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.show()

if "Fare" in eda.columns:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=eda, x="Pclass", y="Fare")
    plt.title("Fare Distribution by Passenger Class")
    plt.show()

**Observations**
- Female passengers had a substantially higher survival rate than male passengers.
- First-class passengers had the highest survival rate, followed by second and third class.
- Fare values are strongly right-skewed and contain high-value outliers; fares are generally higher in first class.

## 6. Relationship analysis: pairplot and correlation heatmap

In [ ]:
numeric_cols = [c for c in ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"] if c in eda.columns]
sns.pairplot(eda[numeric_cols].dropna(), corner=True, diag_kind="hist")
plt.show()

plt.figure(figsize=(9, 6))
sns.heatmap(eda[numeric_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

**Observations**
- Passenger class and fare are related: lower class number generally corresponds to higher fares.
- Fare tends to have a positive association with survival, while Pclass has a negative association with survival because 1 denotes first class and 3 denotes third class.
- Correlation indicates association, not causation. Sex is categorical and is not represented in this numeric-only heatmap.

## 7. Outlier and anomaly checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(y=eda["Age"], ax=axes[0])
axes[0].set_title("Age Outliers")
sns.boxplot(y=eda["Fare"], ax=axes[1])
axes[1].set_title("Fare Outliers")
plt.tight_layout()
plt.show()

# IQR-based count of potential numeric outliers
for col in ["Age", "Fare"]:
    if col in eda.columns:
        q1, q3 = eda[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = ((eda[col] < lower) | (eda[col] > upper)).sum()
        print(f"{col}: {count} potential IQR outliers (bounds {lower:.2f}, {upper:.2f})")

## 8. Key findings and conclusion

1. Survival was not evenly distributed: roughly 38% of passengers in the standard 891-row training sample survived.
2. Sex was a strong differentiator: women had a much higher survival rate than men.
3. Passenger class mattered: first-class passengers had better survival outcomes than second- and third-class passengers.
4. Age and fare distributions contain skewness and potential outliers.
5. Age, Cabin and (to a much smaller extent) Embarked contain missing values in the standard training dataset. Cabin is especially incomplete.
6. The analysis is descriptive. It identifies associations and patterns but does not prove causation.

**Final outcome:** This EDA demonstrates how descriptive statistics, missing-value checks, histograms, boxplots, count plots, scatter/pair plots and a correlation heatmap can be combined to understand a dataset.